In [ ]:
import sys
from pathlib import Path

_SRC = Path.cwd().parent / "src"
if str(_SRC) not in sys.path:
    sys.path.insert(0, str(_SRC))

import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from fraud_detect import config, viz, models, evaluation as ev, tuning
from fraud_detect.models import ModelBackend

warnings.filterwarnings("ignore")
viz.configure_style()

## 13. Model Evaluation & Comparison

Comprehensive evaluation: ROC, PR curves, threshold analysis,
confusion matrices, cumulative gain, and McNemar's test
for statistical significance.

> Logic in `fraud_detect.evaluation` | Plots in `fraud_detect.viz`

In [ ]:
path = config.PROCESSED_TRAIN_PATH if config.PROCESSED_TRAIN_PATH.exists() else config.MERGED_TRAIN_PATH
train = pd.read_parquet(path)
split = models.make_train_val_split(train)
print(f"Train: {split.X_train.shape}, Val: {split.X_val.shape}")

### 13.1 Load Models

In [ ]:
tuned_models = {}
for backend in ModelBackend:
    params = tuning.load_best_params(backend, fallback_to_defaults=True)
    result = models.train_model(train, backend=backend, params=params)
    tuned_models[backend.value] = result

In [ ]:
y_true = split.y_val
y_pred_dict = {name: r.model.predict_proba(split.X_val)[:, 1]
               for name, r in tuned_models.items()}

### 13.2 Evaluation Reports

In [ ]:
reports = {}
for name, preds in y_pred_dict.items():
    report = ev.compute_evaluation(y_true, preds, model_name=name)
    reports[name] = report
    print(f"{name:12s} | AUC: {report.auc:.5f} | F1: {report.f1:.4f} | "
          f"Threshold: {report.best_threshold:.4f} | Youden: {report.youden_index:.4f}")

### 13.3 Model Comparison

In [ ]:
ev.compare_models(reports)

### 13.4 ROC

In [ ]:
y_true_dict = {name: y_true for name in y_pred_dict}
fig, ax = viz.plot_roc_curves(y_true_dict, y_pred_dict)
viz.save_figure(fig, config.METADATA_DIR / "roc_curves_comparison.png")
plt.show()

### 13.5 PR Curves

More informative than ROC for imbalanced data (~3.5% fraud).

In [ ]:
fig, ax = viz.plot_pr_curves(y_true_dict, y_pred_dict)
viz.save_figure(fig, config.METADATA_DIR / "pr_curves_comparison.png")
plt.show()

### 13.6 Confusion Matrices (Optimal Threshold)

In [ ]:
# Plot confusion matrices for each model at its optimal threshold
fig, axes = plt.subplots(1, len(reports), figsize=(5*len(reports), 4))
if len(reports) == 1:
    axes = [axes]

for ax, (name, report) in zip(axes, reports.items(), strict=True):
    _, _ = viz.plot_confusion_matrix(
        y_true,
        (y_pred_dict[name] >= report.best_threshold).astype(int),
        ax=ax,
    )
    ax.set_title(f"{name}\n(thresh={report.best_threshold:.3f})")

plt.tight_layout()
viz.save_figure(fig, config.METADATA_DIR / "confusion_matrices.png")
plt.show()

### 13.7 Metrics Overview

In [ ]:
fig, ax = viz.plot_metrics_comparison(comparison_df)
viz.save_figure(fig, config.METADATA_DIR / "metrics_comparison_bar.png")
plt.show()

### 13.8 Threshold Analysis (Best Model)

In [ ]:
# Pick the best model for threshold analysis
best_model_name = comparison_df.iloc[0]["model"]
print(f"Analysing thresholds for best model: {best_model_name}")

fig, ax = viz.plot_threshold_analysis(y_true, y_pred_dict[best_model_name])
ax.set_title(f"Threshold Analysis — {best_model_name}")
viz.save_figure(fig, config.METADATA_DIR / f"{best_model_name}_threshold_analysis.png")
plt.show()

### 13.9 Cumulative Gain

In [ ]:
fig, ax = viz.plot_cumulative_gain(y_true, y_pred_dict[best_model_name])
ax.set_title(f"Cumulative Gain — {best_model_name}")
viz.save_figure(fig, config.METADATA_DIR / f"{best_model_name}_cumulative_gain.png")
plt.show()

### 13.10 McNemar's Test

Statistical significance between model predictions (p < 0.05).

In [ ]:
model_names = list(y_pred_dict.keys())
for i in range(len(model_names)):
    for j in range(i+1, len(model_names)):
        ni, nj = model_names[i], model_names[j]
        pi = (y_pred_dict[ni] >= reports[ni].best_threshold).astype(int)
        pj = (y_pred_dict[nj] >= reports[nj].best_threshold).astype(int)
        result = ev.mcnemar_test(pi, pj, y_true)
        print(f"{ni:12s} vs {nj:12s} | p={result['p_value']:.5f} | {result['conclusion']}")

### 13.11 Summary

| Best Model | ROC-AUC | PR-AUC | F1 | Threshold |
|---|---|---|---|---|

Next: notebook 14 — error analysis.